In [1]:
import aml_pipeline_methods as aml

In [2]:
#
# Read in the csv data
#
df_train, (df_test2017, df_test2018, df_test2019) = aml.pull_train_test_datasets()

In [3]:
#
# Replace NaN Values with Zero
#
df_train_o = aml.replace_any_nan_values(df_train)
df_test2017_o = aml.replace_any_nan_values(df_test2017)
df_test2018_o = aml.replace_any_nan_values(df_test2018)
df_test2019_o = aml.replace_any_nan_values(df_test2019)

In [5]:
#
# Get the features-label sets for train, validate and test
#
x_train, y_train = aml.get_features_labels_from_df(df_train_o)
x_test2017, y_test2017 = aml.get_features_labels_from_df(df_test2017_o)
x_test2018, y_test2018 = aml.get_features_labels_from_df(df_test2018_o)
x_test2019, y_test2019 = aml.get_features_labels_from_df(df_test2019_o)

INFO: The shape of the feature set is (2891, 927)
INFO: The shape of the labels is (2891,)
INFO: The shape of the feature set is (193, 927)
INFO: The shape of the labels is (193,)
INFO: The shape of the feature set is (192, 927)
INFO: The shape of the labels is (192,)
INFO: The shape of the feature set is (192, 927)
INFO: The shape of the labels is (192,)


In [6]:
#
# Normalize the feature sets
#
x_train_norm, x_test2017_norm = aml.normalize_datasets(x_train, x_test2017)
_, x_test2018_norm = aml.normalize_datasets(x_train, x_test2018)
_, x_test2019_norm = aml.normalize_datasets(x_train, x_test2019)

In [7]:
#
# Run PCA to reduce the features dimension
#
N_PCA_COMPONENTS = 300
pca_, x_train_pca, x_test2017_pca = aml.transform_data_with_pca(N_PCA_COMPONENTS, x_train_norm, x_test2017_norm)

x_test2018_pca = pca_.transform(x_test2018_norm)
print(f'INFO: test 2018 pca shape -> {x_test2018_pca.shape}')

x_test2019_pca = pca_.transform(x_test2019_norm)
print(f'INFO: test 2019 pca shape -> {x_test2019_pca.shape}')

INFO: running PCA for 300 components

INFO: test 2018 pca shape -> (192, 300)
INFO: test 2019 pca shape -> (192, 300)


# Setup the Multilayer Perceptron (MLP) Pipeline

In [9]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.colors import ListedColormap

In [12]:
clf = MLPClassifier(
                   hidden_layer_sizes=(150, 100, 50, 25),
                   verbose=True,
#                   learning_rate_init=0.01
)

clf.fit(x_train_pca, y_train)

Iteration 1, loss = 0.68856781
Iteration 2, loss = 0.60106239
Iteration 3, loss = 0.53217382
Iteration 4, loss = 0.45904303
Iteration 5, loss = 0.37708776
Iteration 6, loss = 0.28699794
Iteration 7, loss = 0.19167345
Iteration 8, loss = 0.10898005
Iteration 9, loss = 0.05234145
Iteration 10, loss = 0.02365561
Iteration 11, loss = 0.01169902
Iteration 12, loss = 0.00687345
Iteration 13, loss = 0.00460085
Iteration 14, loss = 0.00330739
Iteration 15, loss = 0.00258686
Iteration 16, loss = 0.00212307
Iteration 17, loss = 0.00176050
Iteration 18, loss = 0.00149811
Iteration 19, loss = 0.00129824
Iteration 20, loss = 0.00114359
Iteration 21, loss = 0.00101719
Iteration 22, loss = 0.00091193
Iteration 23, loss = 0.00082725
Iteration 24, loss = 0.00075421
Iteration 25, loss = 0.00069486
Iteration 26, loss = 0.00064097
Iteration 27, loss = 0.00059590
Iteration 28, loss = 0.00055629
Iteration 29, loss = 0.00052047
Iteration 30, loss = 0.00048977
Iteration 31, loss = 0.00046278
Iteration 32, los

MLPClassifier(hidden_layer_sizes=(150, 100, 50, 25), verbose=True)

In [13]:
#
# Make a prediction
#
y_pred2017 = clf.predict(x_test2017_pca)

accuracy_score(y_test2017, y_pred2017)

0.6321243523316062

In [14]:
y_pred2018 = clf.predict(x_test2018_pca)
accuracy_score(y_test2018, y_pred2018)

0.5833333333333334

In [15]:
y_pred2019 = clf.predict(x_test2019_pca)
accuracy_score(y_test2019, y_pred2019)

0.5520833333333334

In [20]:
learn_rates = [1 / (10**i) for i in range(1, 6)]
best = dict()
best_score = 0
for lr in learn_rates:
    clf = MLPClassifier(
                       hidden_layer_sizes=(150, 100, 50, 25),
                       #verbose=True,
                       learning_rate_init=lr,
                        max_iter=500
    )

    clf.fit(x_train_pca, y_train)

    #
    # Make a prediction
    #
    y_pred2017 = clf.predict(x_test2017_pca)
    score_2017 = accuracy_score(y_test2017, y_pred2017)

    y_pred2018 = clf.predict(x_test2018_pca)
    score_2018 = accuracy_score(y_test2018, y_pred2018)

    y_pred2019 = clf.predict(x_test2019_pca)
    score_2019 = accuracy_score(y_test2019, y_pred2019)
    
    score_avg = (score_2017 + score_2018 + score_2019)/3.0
    if score_avg > best_score:
        best_score = avg_score
        best['score_2017'] = score_2017
        best['score_2018'] = score_2018
        best['score_2019'] = score_2019
        best['score_avg'] = score_avg
        best['learning_rate'] = lr

print('INFO: the best scores were,')
for k,v in best.items():
    print(f'{k} = {v}')

INFO: the best scores were,
score_2017 = 0.6476683937823834
score_2018 = 0.546875
score_2019 = 0.5729166666666666
score_avg = 0.5891533534830167
learning_rate = 1e-05


C:\Users\cocod\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:582: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
best_test = dict()
best_validate = dict()

best_score_test = 0
best_score_val = 0

# try the above loop again but keep the top five configurations
for l1_neurons in range(100, 201, 25):
    for l2_neurons in range(50, 100, 10):
        for l3_neurons in range(10, 50, 10):
            clf = MLPClassifier(hidden_layer_sizes=(l1_neurons, l2_neurons, l3_neurons),
                       random_state=5,
                       #verbose=True,
                       learning_rate_init=0.01)

            print(f'INFO: ({l1_neurons}, {l2_neurons}, {l3_neurons})')
            clf.fit(x_train_pca, y_train)

            #
            # Make a prediction
            #
            y_pred = clf.predict(x_validate_pca)
            validate_score = accuracy_score(y_validate, y_pred)
            print(f'INFO: validate - {validate_score}')
            
            y_pred = clf.predict(x_test_pca)
            test_score = accuracy_score(y_test, y_pred)
            print(f'INFO: test - {test_score}\n')
            
            if test_score > best_score_test:
                best_test['test_score'] = test_score
                best_test['params'] = (l1_neurons, l2_neurons, l3_neurons)
                best_test['validate_score'] = validate_score
                best_score_test = test_score
            
            if validate_score > best_score_val:
                best_validate['validate_score'] = validate_score
                best_validate['params'] = (l1_neurons, l2_neurons, l3_neurons)
                best_validate['test_score'] = test_score
                best_score_val = validate_score
                
print(f'INFO: best test -> {best_test}\n')
print(f'INFO: best validate -> {best_validate}\n')

# Use GridSearchCV for hyper-parameter optimization

In [23]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

N_PCA_COMPONENTS = [150, 200, 250, 300, 350]
clf_dict = dict()

mlp_architectures = [
    (150, 100, 50, 25),
    (200, 100, 50, 25),
    (300, 150, 75, 25),
    (100, 50, 25, 10),
    (150, 75, 25, 10),
    (300, 150, 75),
    (200, 100, 50),
    (100, 50, 25),
    (150, 75, 25),
    (50, 25, 10)
]

best_score = 0
best = dict()

for N in N_PCA_COMPONENTS:
    #
    # Run PCA to reduce the features dimension
    #
    pca_, x_train_pca, x_test2017_pca = aml.transform_data_with_pca(N, x_train_norm, x_test2017_norm)

    x_test2018_pca = pca_.transform(x_test2018_norm)
    print(f'INFO: test 2018 pca shape -> {x_test2018_pca.shape}')

    x_test2019_pca = pca_.transform(x_test2019_norm)
    print(f'INFO: test 2019 pca shape -> {x_test2019_pca.shape}')

    #
    # Setup the MLP Classifier
    #

    mlp = MLPClassifier()

    parameter_space = {
        'hidden_layer_sizes': mlp_architectures,
        'activation': ['tanh', 'relu'],
        'solver': ['sgd', 'adam'],
        'alpha': [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.05, 0.1],
        'learning_rate': ['constant', 'adaptive']
    }

    #
    # Run the search
    #
    clf = GridSearchCV(mlp, parameter_space, n_jobs=4, cv=3)
    clf.fit(x_train_pca, y_train)

    #
    # See the best results
    #
    print('Best parameters found:\n', clf.best_params_)

    # All results
    means = clf.cv_results_['mean_test_score']
    stds = clf.cv_results_['std_test_score']

    #
    # Run the classifier on the test set
    #
    y_pred2017 = clf.predict(x_test2017_pca)
    clf_report2017 = classification_report(y_test2017, y_pred2017)
    score_2017 = clf.score(x_test2017_pca, y_test2017)
    
    y_pred2018 = clf.predict(x_test2018_pca)
    clf_report2018 = classification_report(y_test2018, y_pred2018)
    score_2018 = clf.score(x_test2018_pca, y_test2018)
    
    y_pred2019 = clf.predict(x_test2019_pca)
    clf_report2019 = classification_report(y_test2019, y_pred2019)
    score_2019 = clf.score(x_test2019_pca, y_test2019)
    
    score_avg = (score_2017 + score_2018 + score_2019) / 3.0
    
    dtemp = dict()
    dtemp['best_clf_params'] = clf.best_params_
    dtemp['score_avg'] = score_avg
    dtemp['score_2017'] = score_2017
    dtemp['score_2018'] = score_2018
    dtemp['score_2019'] = score_2019
    dtemp['N_components'] = N
    
    best[N] = dtemp
    
    print('\n' + '*'*80 + '\n')
    

INFO: running PCA for 150 components

INFO: test 2018 pca shape -> (192, 150)
INFO: test 2019 pca shape -> (192, 150)
Best parameters found:
 {'activation': 'tanh', 'alpha': 0.05, 'hidden_layer_sizes': (150, 75, 25, 10), 'learning_rate': 'adaptive', 'solver': 'adam'}

********************************************************************************

INFO: running PCA for 200 components

INFO: test 2018 pca shape -> (192, 200)
INFO: test 2019 pca shape -> (192, 200)


C:\Users\cocod\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:582: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Best parameters found:
 {'activation': 'tanh', 'alpha': 0.1, 'hidden_layer_sizes': (200, 100, 50), 'learning_rate': 'constant', 'solver': 'sgd'}

********************************************************************************

INFO: running PCA for 250 components

INFO: test 2018 pca shape -> (192, 250)
INFO: test 2019 pca shape -> (192, 250)


C:\Users\cocod\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:582: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Best parameters found:
 {'activation': 'tanh', 'alpha': 0.05, 'hidden_layer_sizes': (50, 25, 10), 'learning_rate': 'adaptive', 'solver': 'sgd'}

********************************************************************************

INFO: running PCA for 300 components

INFO: test 2018 pca shape -> (192, 300)
INFO: test 2019 pca shape -> (192, 300)
Best parameters found:
 {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (300, 150, 75, 25), 'learning_rate': 'adaptive', 'solver': 'adam'}

********************************************************************************

INFO: running PCA for 350 components

INFO: test 2018 pca shape -> (192, 350)
INFO: test 2019 pca shape -> (192, 350)
Best parameters found:
 {'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (150, 75, 25), 'learning_rate': 'adaptive', 'solver': 'adam'}

********************************************************************************



In [27]:
best_score = 0
best_d = dict()

for d in best.values():
    if d['score_avg'] > best_score:
        best_score = d['score_avg']
        best_d = d

for k,v in best_d.items():
    print(f'{k}: {v}')

best_clf_params: {'activation': 'tanh', 'alpha': 0.05, 'hidden_layer_sizes': (150, 75, 25, 10), 'learning_rate': 'adaptive', 'solver': 'adam'}
score_avg: 0.6169671128382269
score_2017: 0.6269430051813472
score_2018: 0.625
score_2019: 0.5989583333333334
N_components: 150


In [30]:
import os
import pandas as pd

stats_dir = r'C:\Users\cocod\Work\Repos\NFL_stats_tracker\stats'

# use the best classifier parameters
N = 150


df_train, (df_test2017, df_test2018, df_test2019) = aml.pull_train_test_datasets()


#
# Replace NaN Values with Zero
#
df_train_o = aml.replace_any_nan_values(df_train)
df_test2017_o = aml.replace_any_nan_values(df_test2017)
df_test2018_o = aml.replace_any_nan_values(df_test2018)
df_test2019_o = aml.replace_any_nan_values(df_test2019)

#
# Get the features-label sets for train, validate and test
#
x_train, y_train = aml.get_features_labels_from_df(df_train_o)
x_test2017, y_test2017 = aml.get_features_labels_from_df(df_test2017_o)
x_test2018, y_test2018 = aml.get_features_labels_from_df(df_test2018_o)
x_test2019, y_test2019 = aml.get_features_labels_from_df(df_test2019_o)

#
# Normalize the feature sets
#
x_train_norm, x_test2017_norm = aml.normalize_datasets(x_train, x_test2017)
_, x_test2018_norm = aml.normalize_datasets(x_train, x_test2018)
_, x_test2019_norm = aml.normalize_datasets(x_train, x_test2019)

# 
# Run PCA
#
pca_, x_train_pca, x_test2017_pca = aml.transform_data_with_pca(N, x_train_norm, x_test2017_norm)
    
x_test2018_pca = pca_.transform(x_test2018_norm)
print(f'INFO: test 2018 pca shape -> {x_test2018_pca.shape}')

x_test2019_pca = pca_.transform(x_test2019_norm)
print(f'INFO: test 2019 pca shape -> {x_test2019_pca.shape}')

# train the classifier

# TODO
hidden_layers = (150, 75, 25, 10)
learning_rate = 'adaptive'
activation = 'tanh'
solver = 'adam'
alpha=0.05

clf_mlp = MLPClassifier(hidden_layer_sizes=hidden_layers,
           verbose=True,
           learning_rate=learning_rate,
           activation=activation,
            solver=solver,
            alpha=alpha)

clf_mlp.fit(x_train_pca, y_train)


test_year_sets = [
    (2017, x_test2017_pca, y_test2017, df_test2017),
    (2018, x_test2018_pca, y_test2018, df_test2018),
    (2019, x_test2019_pca, y_test2019, df_test2019)
]
for year, x_test_pca, y_test, df_test in test_year_sets:
    # Determine the accuracy prediction for point spread bets
    total_wins_wts = 0

    for ix in range(len(x_test_pca)):
        X_i = x_test_pca[ix, :]
        y_pred = clf_mlp.predict(X_i.reshape((1,N)))

        if 0 == y_pred:
            # get the spread for the away team
            team = df_test.iloc[ix]['Away Team']
        else:
            # get the spread for the home team
            team = df_test.iloc[ix]['Home Team']

        week_n = df_test.iloc[ix]['Week']

        # open team dataframe
        team_csv = os.path.join(stats_dir, f'{year}', 'team_season_stats', f'{team}_season_{year}.csv')
        df = pd.read_csv(team_csv)

        mask = df['Week'] == week_n
        point_spread = int(df[mask]['GI Vegas Line'].values[0])

        team_score = int(df[mask]['LS Final Score'].values[0])
        opponent_score = int(df[mask]['LS Total Points Gave Up'])

        y_true = y_test[ix]

        if point_spread > 0:
            # underdog team
            if y_pred == y_true:
                # underdog won the game so won the bet
                total_wins_wts += 1
            else:
                # check how much lost by
                diff = abs(team_score - opponent_score)
                if diff < point_spread:
                    total_wins_wts += 1
        else: # point_spread <= 0
            # favored team
            if y_pred == y_true:
                # the favored team won the game, check the point spread was met
                diff = abs(team_score - opponent_score)
                if diff > abs(point_spread):
                    # the point spread was met
                    total_wins_wts += 1

    wts_win_rate = total_wins_wts / len(x_test_pca)
    print(f'Year {year} - {wts_win_rate*100:.2f}%')

INFO: The shape of the feature set is (2891, 927)
INFO: The shape of the labels is (2891,)
INFO: The shape of the feature set is (193, 927)
INFO: The shape of the labels is (193,)
INFO: The shape of the feature set is (192, 927)
INFO: The shape of the labels is (192,)
INFO: The shape of the feature set is (192, 927)
INFO: The shape of the labels is (192,)
INFO: running PCA for 150 components

INFO: test 2018 pca shape -> (192, 150)
INFO: test 2019 pca shape -> (192, 150)
Iteration 1, loss = 0.74157142
Iteration 2, loss = 0.63859399
Iteration 3, loss = 0.59122143
Iteration 4, loss = 0.54812837
Iteration 5, loss = 0.49825375
Iteration 6, loss = 0.44048976
Iteration 7, loss = 0.37436050
Iteration 8, loss = 0.30024480
Iteration 9, loss = 0.22647710
Iteration 10, loss = 0.16299549
Iteration 11, loss = 0.11775907
Iteration 12, loss = 0.09031878
Iteration 13, loss = 0.07402049
Iteration 14, loss = 0.06563300
Iteration 15, loss = 0.06084493
Iteration 16, loss = 0.05796463
Iteration 17, loss = 

In [ ]:
accuracy_score(y_true, y_pred)

In [ ]:
best_clf = clf.best_estimator_
y_pred = best_clf.predict(x_test_pca)
print(accuracy_score(y_true, y_pred))

In [ ]:
best_clf.get_params()

In [ ]:
def run_mlp_nfl_predict(iX_train:np.ndarray, iY_train:np.ndarray,
                        iX_validate:np.ndarray, iY_validate:np.ndarray,
                        iX_test:np.ndarray, iY_test:np.ndarray):
    pass